# Full Fine-Tuning an Automobile Insurance Claim Intake Assistant

### Introduction

In this milestone, I continued building the automobile insurance claim intake assistant from my earlier projects. The assistant is meant to extract key details from accident descriptions so claims representatives can review information more efficiently. It is not meant to decide fault, liability, or coverage. The question I’m testing is narrower: can a language model accurately extract information from a customer’s account of an accident?

In Milestone 2, I tested OpenAI’s API to see how well a generative model could interpret accident narratives. My professor’s feedback raised concerns about fine-tuning OpenAI models, so I changed approach. I chose SmolLM2-135M-Instruct, an open-weight model small enough to fully fine-tune in Google Colab. This allowed me to update all of the model's parameters rather than using LoRA, which only trains a smaller set of additional parameters.

This notebook walks through the whole process: building a synthetic dataset, evaluating the original model, fully fine-tuning it, and testing the results. I’m most interested in whether fine-tuning improves the model’s ability to return structured responses and correctly identify the collision type, vehicle damage, and drivability.

I also want to look closely at where the experiment falls short. A model can do well in training and still fail on accident descriptions it hasn’t seen. By comparing the original and fine-tuned models, I hope to understand the difference between learning a response format and extracting the right information.

In [23]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Check Colab runtime settings.")


PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [24]:
%pip install -q -U transformers datasets accelerate


In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

model.to("cuda")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Percentage trainable: {trainable_params / total_params * 100:.2f}%")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model: HuggingFaceTB/SmolLM2-135M-Instruct
Total parameters: 134,515,008
Trainable parameters: 134,515,008
Percentage trainable: 100.00%


## Model Selection and Initial Setup

For this milestone, I decided to use SmolLM2-135M-Instruct because I needed a model that I could fully fine-tune instead of relying on prompt engineering alone. My earlier experiments used OpenAI's API, but my professor pointed out that fine-tuning availability could become a limitation. I chose a smaller open-weight model so I could work directly with its parameters using Google Colab.

Before beginning training, I checked whether Colab could access a GPU. The environment detected an NVIDIA Tesla T4, which I plan to use for the experiment. I also included a parameter count to confirm that the model's weights are trainable. This will be important when I evaluate whether the training process actually meets the full fine-tuning requirement.

## Baseline Evaluation Before Fine-Tuning

Before training the model, I wanted to understand how it would respond to an automobile insurance claim without any additional task-specific training. I created a sample accident narrative containing several details that an intake assistant should identify, including the type of collision, vehicle damage, and missing insurance information.

I also instructed the model not to make assumptions about fault, liability, or coverage. This is important because the purpose of my application is to organize information for a claims representative, not to make decisions that require additional investigation.

I plan to use this response as a baseline and compare it with the model's output after fine-tuning. This should help me determine whether training improves the model's ability to extract relevant information without introducing unsupported details.

In [26]:
import torch

def generate_claim_response(claim_text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an automobile insurance claim intake assistant. "
                "Extract only information explicitly provided. "
                "Do not invent facts or determine fault, liability, "
                "or insurance coverage."
            )
        },
        {
            "role": "user",
            "content": claim_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


test_claim = (
    "I was driving home yesterday when another vehicle "
    "hit the rear of my car at a red light. "
    "My bumper is damaged, but I was able to drive home. "
    "The other driver gave me their name, but I didn't "
    "get their insurance information."
)

baseline_response = generate_claim_response(test_claim)

print("ORIGINAL MODEL RESPONSE:\n")
print(baseline_response)


ORIGINAL MODEL RESPONSE:

It sounds like you were driving home when the other vehicle hit the rear of your car at a red light. Unfortunately, the damage to your bumper is not a guarantee of liability, but it does suggest that you may have been involved in an accident.

To determine your fault, liability, and insurance coverage, I would need more information about the accident. Could you please provide me with the details of the accident, including the time, location, and any witnesses who may have been present? Additionally, could you please provide me with the insurance information of the other driver, including their name, insurance company, and any relevant details?


## Initial Baseline Results

The original model recognized that the accident involved a rear-end collision and identified damage to the vehicle's bumper. It also requested additional insurance information, which was missing from the accident description. However, I noticed that it overlooked an important detail: the driver stated that the vehicle was still drivable.

Another issue was that the model asked for the other driver's name even though the narrative already indicated that the name had been collected. The response also discussed determining fault, liability, and coverage, which are outside the intended responsibilities of my intake assistant.

Based on this first experiment, the model appears capable of understanding the general situation, but it does not consistently extract all the relevant details. It also responds conversationally rather than organizing the information into a structured intake record. These are the behaviors I want to improve through fine-tuning.

This is only one baseline example, so I cannot conclude that the model performs poorly across all claims. I will need additional test cases to evaluate whether fine-tuning produces consistent improvements.

# Training Dataset Preparation

For this stage of the project, I am creating synthetic automobile accident narratives with corresponding structured intake records. I chose synthetic examples because they allow me to experiment without using actual customers' personal or insurance information.

Each example contains a description of an accident and an expected response showing what the model should extract. I want the model to learn how to identify information that is explicitly stated, recognize missing details, and avoid making assumptions about liability or coverage.

I will begin with a small dataset to confirm that the formatting works before expanding it for training. I also plan to separate training, validation, and test examples so that I can evaluate the model using accident descriptions it has not encountered during training.

In [27]:
import json
import pandas as pd

claim_examples = [
    {
        "narrative": (
            "I was stopped at a red light when another car "
            "hit me from behind. My bumper was damaged, "
            "but I drove home. I got the other driver's name "
            "but not their insurance information."
        ),
        "collision_type": "Rear-end collision",
        "damage": "Bumper damage",
        "drivable": "Yes",
        "missing_information": [
            "Other driver's insurance information"
        ]
    },
    {
        "narrative": (
            "Someone backed into my parked car at the grocery "
            "store. The passenger door is dented. I did not "
            "see the accident happen."
        ),
        "collision_type": "Parking lot collision",
        "damage": "Passenger door dent",
        "drivable": "Unknown",
        "missing_information": [
            "Other driver's identity",
            "Other driver's insurance information"
        ]
    },
    {
        "narrative": (
            "While changing lanes, my car collided with "
            "another vehicle. Both cars pulled over. "
            "My side mirror broke, and nobody reported injuries."
        ),
        "collision_type": "Lane-change collision",
        "damage": "Broken side mirror",
        "drivable": "Unknown",
        "missing_information": [
            "Other driver's insurance information",
            "Vehicle drivability"
        ]
    },
    {
        "narrative": (
            "A deer ran into the road, and I hit it. "
            "The front of my car is damaged, and the "
            "vehicle had to be towed."
        ),
        "collision_type": "Animal collision",
        "damage": "Front-end damage",
        "drivable": "No",
        "missing_information": [
            "Exact accident location"
        ]
    },
    {
        "narrative": (
            "My vehicle was sideswiped while parked on "
            "the street. The other driver left a note "
            "with their name and phone number. "
            "There is damage to the driver's side."
        ),
        "collision_type": "Sideswipe collision",
        "damage": "Driver-side damage",
        "drivable": "Unknown",
        "missing_information": [
            "Other driver's insurance information",
            "Vehicle drivability"
        ]
    }
]

df = pd.DataFrame(claim_examples)

print("Number of examples:", len(df))
display(df)

Number of examples: 5


,narrative,collision_type,damage,drivable,missing_information
0,"I was stopped at a red light when another car hit me from behind. My bumper was damaged, but I d...",Rear-end collision,Bumper damage,Yes,[Other driver's insurance information]
1,Someone backed into my parked car at the grocery store. The passenger door is dented. I did not ...,Parking lot collision,Passenger door dent,Unknown,"[Other driver's identity, Other driver's insurance information]"
2,"While changing lanes, my car collided with another vehicle. Both cars pulled over. My side mirro...",Lane-change collision,Broken side mirror,Unknown,"[Other driver's insurance information, Vehicle drivability]"
3,"A deer ran into the road, and I hit it. The front of my car is damaged, and the vehicle had to b...",Animal collision,Front-end damage,No,[Exact accident location]
4,My vehicle was sideswiped while parked on the street. The other driver left a note with their na...,Sideswipe collision,Driver-side damage,Unknown,"[Other driver's insurance information, Vehicle drivability]"


In [28]:
# Check for missing values and duplicate narratives
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate narratives:")
print(df["narrative"].duplicated().sum())

print("\nCollision types:")
print(df["collision_type"].value_counts())

print("\nDrivability labels:")
print(df["drivable"].value_counts())


Missing values:
narrative              0
collision_type         0
damage                 0
drivable               0
missing_information    0
dtype: int64

Duplicate narratives:
0

Collision types:
collision_type
Rear-end collision       1
Parking lot collision    1
Lane-change collision    1
Animal collision         1
Sideswipe collision      1
Name: count, dtype: int64

Drivability labels:
drivable
Unknown    3
Yes        1
No         1
Name: count, dtype: int64


## Expanding the Synthetic Dataset

My initial data quality check found no missing values or duplicate narratives in the five examples. However, five examples would not provide enough variety for a meaningful fine-tuning experiment.

To expand the dataset, I decided to create additional synthetic accident narratives using different collision types, damage descriptions, and vehicle drivability conditions. I want the model to learn how to extract these details consistently rather than memorize one particular accident scenario.

I am also keeping the data synthetic to avoid introducing actual customer information into the training process. After generating the examples, I will inspect the dataset again and separate the records into training, validation, and test groups.

In [29]:
import pandas as pd
import random

random.seed(42)

scenarios = [
    ("Rear-end collision", "Another vehicle struck the rear of my car."),
    ("Parking lot collision", "A vehicle backed into my car in a parking lot."),
    ("Lane-change collision", "Another vehicle hit my car while changing lanes."),
    ("Animal collision", "I struck a deer that ran into the road."),
    ("Sideswipe collision", "Another vehicle sideswiped my car.")
]

damages = [
    ("Bumper damage", "The bumper was damaged."),
    ("Passenger door dent", "The passenger door was dented."),
    ("Broken side mirror", "The side mirror was broken."),
    ("Front-end damage", "The front of the vehicle was damaged."),
    ("Driver-side damage", "The driver's side was damaged.")
]

drivability = [
    ("Yes", "I was able to drive the vehicle afterward."),
    ("No", "The vehicle could not be driven and required towing."),
    ("Unknown", "I have not confirmed whether the vehicle is drivable.")
]

records = []

for collision_label, collision_text in scenarios:
    for damage_label, damage_text in damages:
        for drivable_label, drivable_text in drivability:
            narrative = (
                collision_text + " " +
                damage_text + " " +
                drivable_text
            )

            records.append({
                "narrative": narrative,
                "collision_type": collision_label,
                "damage": damage_label,
                "drivable": drivable_label
            })

expanded_df = pd.DataFrame(records)

print("Generated examples:", len(expanded_df))
print("Duplicate narratives:", expanded_df["narrative"].duplicated().sum())

display(expanded_df.head(10))


Generated examples: 75
Duplicate narratives: 0


,narrative,collision_type,damage,drivable
0,Another vehicle struck the rear of my car. The bumper was damaged. I was able to drive the vehic...,Rear-end collision,Bumper damage,Yes
1,Another vehicle struck the rear of my car. The bumper was damaged. The vehicle could not be driv...,Rear-end collision,Bumper damage,No
2,Another vehicle struck the rear of my car. The bumper was damaged. I have not confirmed whether ...,Rear-end collision,Bumper damage,Unknown
3,Another vehicle struck the rear of my car. The passenger door was dented. I was able to drive th...,Rear-end collision,Passenger door dent,Yes
4,Another vehicle struck the rear of my car. The passenger door was dented. The vehicle could not ...,Rear-end collision,Passenger door dent,No
5,Another vehicle struck the rear of my car. The passenger door was dented. I have not confirmed w...,Rear-end collision,Passenger door dent,Unknown
6,Another vehicle struck the rear of my car. The side mirror was broken. I was able to drive the v...,Rear-end collision,Broken side mirror,Yes
7,Another vehicle struck the rear of my car. The side mirror was broken. The vehicle could not be ...,Rear-end collision,Broken side mirror,No
8,Another vehicle struck the rear of my car. The side mirror was broken. I have not confirmed whet...,Rear-end collision,Broken side mirror,Unknown
9,Another vehicle struck the rear of my car. The front of the vehicle was damaged. I was able to d...,Rear-end collision,Front-end damage,Yes


## Preparing the Data for Fine-Tuning

After expanding the dataset, I generated 75 synthetic accident narratives without duplicate records. Each narrative contains a collision description, reported damage, and information about whether the vehicle could be driven.

I recognize that the examples use similar sentence structures, which could make the task easier for the model. For that reason, I plan to evaluate the trained model on additional narratives written differently from the training examples.

Before fine-tuning, I need to convert each record into the conversational format expected by SmolLM2. The model will receive an accident narrative and learn to produce a structured response containing the relevant claim details.

I will also separate a portion of the generated records for validation so I can monitor how well the model performs on examples not directly used for weight updates.

In [30]:
import json
from sklearn.model_selection import train_test_split
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an automobile insurance claim intake assistant. "
    "Extract only facts explicitly stated in the narrative. "
    "Return a JSON object with collision_type, damage, "
    "and drivable. Do not determine fault, liability, "
    "or insurance coverage."
)

def format_example(row):
    expected_output = {
        "collision_type": row["collision_type"],
        "damage": row["damage"],
        "drivable": row["drivable"]
    }

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["narrative"]},
        {
            "role": "assistant",
            "content": json.dumps(expected_output)
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

train_df, val_df = train_test_split(
    expanded_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_texts = [
    format_example(row)
    for _, row in train_df.iterrows()
]

val_texts = [
    format_example(row)
    for _, row in val_df.iterrows()
]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

print("\nExample formatted training record:\n")
print(train_dataset[0]["text"])


Training examples: 60
Validation examples: 15

Example formatted training record:

<|im_start|>system
You are an automobile insurance claim intake assistant. Extract only facts explicitly stated in the narrative. Return a JSON object with collision_type, damage, and drivable. Do not determine fault, liability, or insurance coverage.<|im_end|>
<|im_start|>user
A vehicle backed into my car in a parking lot. The side mirror was broken. The vehicle could not be driven and required towing.<|im_end|>
<|im_start|>assistant
{"collision_type": "Parking lot collision", "damage": "Broken side mirror", "drivable": "No"}<|im_end|>



## Tokenization and Full-Parameter Fine-Tuning

After formatting the dataset, I had 60 training examples and 15 validation examples. Each record contains a system instruction, an accident narrative, and a structured JSON response representing the expected claim details.

The next step is tokenization, which converts the text into numerical tokens that the model can process. I also need to identify which tokens belong to the expected assistant response. I want the training process to focus on predicting those response tokens rather than treating the entire conversation as the target.

For this experiment, I am using full-parameter fine-tuning rather than LoRA. This means the optimizer will be allowed to update all 134,515,008 parameters in the model. I will monitor training and validation loss to see whether the model is learning the examples and whether its performance on held-out records changes.

In [31]:
from datasets import Dataset
import torch

MAX_LENGTH = 256

def tokenize_record(example):
    text = example["text"]

    # Locate the assistant response in the chat template
    assistant_marker = "<|im_start|>assistant\n"
    marker_pos = text.rfind(assistant_marker)

    if marker_pos == -1:
        raise ValueError("Assistant marker not found")

    prompt_text = text[:marker_pos + len(assistant_marker)]

    full_tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    labels = input_ids.copy()
    prompt_length = len(prompt_tokens["input_ids"])

    labels[:prompt_length] = [-100] * prompt_length

    if all(label == -100 for label in labels):
        raise ValueError(
            "Assistant response was truncated. Increase MAX_LENGTH."
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

tokenized_train = train_dataset.map(
    tokenize_record,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_record,
    remove_columns=["text"]
)

print("Tokenized training records:", len(tokenized_train))
print("Tokenized validation records:", len(tokenized_val))

sample = tokenized_train[0]
print("Input token count:", len(sample["input_ids"]))
print(
    "Assistant target tokens:",
    sum(label != -100 for label in sample["labels"])
)


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenized training records: 60
Tokenized validation records: 15
Input token count: 117
Assistant target tokens: 31


In [32]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq

model.config.use_cache = False

# Confirm that all parameters remain trainable
for parameter in model.parameters():
    parameter.requires_grad = True

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total = sum(p.numel() for p in model.parameters())

assert trainable == total, "Some parameters are frozen!"

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir="./smollm2_claim_intake",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator
)

print("Training configuration created successfully.")
print(f"Trainable parameters: {trainable:,} / {total:,}")


Training configuration created successfully.
Trainable parameters: 134,515,008 / 134,515,008


In [33]:
for split_name, dataset in [
    ("Training", tokenized_train),
    ("Validation", tokenized_val)
]:
    target_counts = [
        sum(label != -100 for label in record["labels"])
        for record in dataset
    ]

    assert min(target_counts) > 0, (
        f"{split_name} contains an example with no target tokens!"
    )

    print(
        f"{split_name}: {len(dataset)} examples, "
        f"target tokens per example: "
        f"{min(target_counts)}–{max(target_counts)}"
    )

print("\nAll examples contain assistant training targets.")


Training: 60 examples, target tokens per example: 28–32
Validation: 15 examples, target tokens per example: 29–32

All examples contain assistant training targets.


## Running the Full Fine-Tuning Experiment

At this stage, I have prepared 60 training examples and 15 validation examples. I confirmed that all 134,515,008 model parameters are trainable, which is important because this assignment requires full fine-tuning rather than LoRA.

I configured the training process for three epochs using a learning rate of 0.00002. An epoch represents one complete pass through the training dataset. During training, the model will compare its predicted response tokens against the expected JSON responses and calculate a loss value. Backpropagation will then calculate gradients, and the optimizer will use those gradients to update the model's weights.

I am using a separate validation dataset to monitor how the model performs on records that are not used for weight updates. I will compare training and validation loss to look for signs of learning or possible overfitting. Since my dataset is relatively small and uses repeated sentence patterns, I will be cautious about interpreting improvements as evidence that the model would perform equally well on real insurance claims.

In [34]:
import time

print("Starting full-parameter fine-tuning...")

start_time = time.time()

training_result = trainer.train()

elapsed_time = time.time() - start_time

print("\nTraining completed!")
print(f"Training time: {elapsed_time / 60:.2f} minutes")
print(f"Final training loss: {training_result.training_loss:.4f}")

print("\nValidation results:")
evaluation_results = trainer.evaluate()

for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")


Starting full-parameter fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.892595,0.213005
2,0.131759,0.042383
3,0.040840,0.015819


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!
Training time: 1.45 minutes
Final training loss: 0.2836

Validation results:


Training Loss,Validation Loss,Epoch
0.040840,0.015819,3


eval_loss: 0.015818972140550613


## Evaluating the Fine-Tuned Model

The full fine-tuning process completed successfully after three epochs. Training loss decreased from approximately 0.893 during the first epoch to 0.041 during the third epoch. Validation loss also decreased from approximately 0.213 to 0.016.

These results suggest that the model learned the response patterns in my synthetic dataset. However, the training examples used similar sentence structures, so I cannot assume the model will perform equally well on unfamiliar accident descriptions.

My next step is to test the fine-tuned model using the original accident narrative from my baseline experiment. I want to see whether it now produces a structured intake response, identifies the vehicle's drivability, and avoids introducing information that was not provided.

I will also need to evaluate additional unseen examples because one successful response would not be enough to establish reliable performance.

In [35]:
# Test the fine-tuned model on the original baseline claim

fine_tuned_response = generate_claim_response(test_claim)

print("ORIGINAL BASELINE RESPONSE:\n")
print(baseline_response)

print("\n" + "=" * 60 + "\n")

print("FINE-TUNED MODEL RESPONSE:\n")
print(fine_tuned_response)


ORIGINAL BASELINE RESPONSE:

It sounds like you were driving home when the other vehicle hit the rear of your car at a red light. Unfortunately, the damage to your bumper is not a guarantee of liability, but it does suggest that you may have been involved in an accident.

To determine your fault, liability, and insurance coverage, I would need more information about the accident. Could you please provide me with the details of the accident, including the time, location, and any witnesses who may have been present? Additionally, could you please provide me with the insurance information of the other driver, including their name, insurance company, and any relevant details?


FINE-TUNED MODEL RESPONSE:

Your text provides a clear explanation of the incident, but it could be improved for clarity, grammar, and concision. Here's a revised version:

"I was driving home from work yesterday when another vehicle hit the rear of my car at a red light. My bumper was damaged, but I managed to driv

## Investigating Unexpected Model Behavior

My first test after fine-tuning did not produce the structured JSON response I expected. Instead, the model attempted to rewrite the accident description and introduced a detail about driving home from work that was not included in the original narrative.

This was an important finding because the training and validation loss had decreased substantially. The result demonstrated that low loss does not automatically mean the model will perform correctly on unfamiliar inputs.

I also noticed that my original testing function used a different system instruction from the one used during training. To investigate whether this contributed to the unexpected behavior, I decided to repeat the test using the same system prompt that appeared in my training examples.

This additional experiment should help me determine whether the model learned the intended extraction format or whether further changes to the training data and evaluation process are necessary.

In [36]:
def generate_structured_claim(claim_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": claim_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        response_tokens,
        skip_special_tokens=True
    )


structured_response = generate_structured_claim(test_claim)

print("FINE-TUNED MODEL WITH TRAINING PROMPT:\n")
print(structured_response)


FINE-TUNED MODEL WITH TRAINING PROMPT:

{"collision_type": "Red light collision", "damage": "Bumper damage", "drivable": "Unknown"}


## Structured Response Evaluation

When I repeated the experiment using the same system prompt from training, the model produced a JSON response instead of rewriting the accident narrative. This suggests that the model learned the expected output format, but the extracted information was not entirely accurate.

The model correctly identified bumper damage. However, it classified the accident as a red light collision instead of a rear-end collision. It also labeled the vehicle's drivability as unknown, even though the narrative stated that the driver was able to drive home.

Only one of the three extracted fields matched my expected answers. This result helped me understand that producing correctly formatted output is different from extracting information accurately. Although training and validation loss decreased significantly, the model still struggled with a narrative written differently from the examples used during training.

I believe the repetitive structure of my synthetic dataset may have contributed to this limitation. The model appears to have learned the format of the task, but it may need more varied examples to recognize the same information when customers describe accidents using different wording.

# Independent Model Evaluation

After testing the fine-tuned model on my original accident narrative, I found that it could produce the expected JSON format but still made mistakes when extracting information. It correctly identified the vehicle damage but misclassified the collision and overlooked that the vehicle was drivable.

To investigate whether this was an isolated problem, I created a separate set of synthetic accident narratives for testing. These examples use different wording from the training data and include situations where vehicle drivability is either explicitly stated or not provided.

I will compare the model's predictions with the expected answers for collision type, damage, and drivability. This will allow me to calculate field-level accuracy and identify which types of information the model handles reliably.

Since the evaluation dataset is small, I will treat the results as an initial experiment rather than evidence that the model is ready for use in an actual insurance claims process.

In [37]:
import pandas as pd

test_cases = [
    {
        "narrative": "I was waiting at a stop sign when a truck crashed into the back of my car. My bumper is cracked, but I drove away.",
        "collision_type": "Rear-end collision",
        "damage": "Bumper damage",
        "drivable": "Yes"
    },
    {
        "narrative": "A driver reversed out of a parking space and hit my passenger door. It left a dent. I haven't checked if the car can be driven.",
        "collision_type": "Parking lot collision",
        "damage": "Passenger door dent",
        "drivable": "Unknown"
    },
    {
        "narrative": "While merging onto the highway, another car struck my side mirror. The mirror broke, but my car still runs and drives.",
        "collision_type": "Lane-change collision",
        "damage": "Broken side mirror",
        "drivable": "Yes"
    },
    {
        "narrative": "A deer jumped in front of me on a rural road. The front of my vehicle was crushed, and a tow truck took it away because it wouldn't move.",
        "collision_type": "Animal collision",
        "damage": "Front-end damage",
        "drivable": "No"
    },
    {
        "narrative": "Someone scraped along the driver's side of my parked car. I noticed the damage when I returned.",
        "collision_type": "Sideswipe collision",
        "damage": "Driver-side damage",
        "drivable": "Unknown"
    },
    {
        "narrative": "I was sitting at a traffic light when a vehicle rear-ended me. The back bumper is damaged and the car won't start.",
        "collision_type": "Rear-end collision",
        "damage": "Bumper damage",
        "drivable": "No"
    },
    {
        "narrative": "Another driver backed into my car outside the supermarket, denting the passenger door. I was able to drive home afterward.",
        "collision_type": "Parking lot collision",
        "damage": "Passenger door dent",
        "drivable": "Yes"
    },
    {
        "narrative": "A car moved into my lane and clipped my mirror. The mirror shattered. I don't know if there are other mechanical problems.",
        "collision_type": "Lane-change collision",
        "damage": "Broken side mirror",
        "drivable": "Unknown"
    },
    {
        "narrative": "I hit a deer on my way home. My vehicle has front-end damage, but I managed to drive it to my house.",
        "collision_type": "Animal collision",
        "damage": "Front-end damage",
        "drivable": "Yes"
    },
    {
        "narrative": "A passing vehicle sideswiped the driver's side of my car. The door was damaged and the vehicle had to be towed because it wasn't safe to drive.",
        "collision_type": "Sideswipe collision",
        "damage": "Driver-side damage",
        "drivable": "No"
    }
]

test_df = pd.DataFrame(test_cases)

print("Independent test examples:", len(test_df))
print("Duplicate test narratives:", test_df["narrative"].duplicated().sum())

display(test_df)

Independent test examples: 10
Duplicate test narratives: 0


,narrative,collision_type,damage,drivable
0,"I was waiting at a stop sign when a truck crashed into the back of my car. My bumper is cracked,...",Rear-end collision,Bumper damage,Yes
1,A driver reversed out of a parking space and hit my passenger door. It left a dent. I haven't ch...,Parking lot collision,Passenger door dent,Unknown
2,"While merging onto the highway, another car struck my side mirror. The mirror broke, but my car ...",Lane-change collision,Broken side mirror,Yes
3,"A deer jumped in front of me on a rural road. The front of my vehicle was crushed, and a tow tru...",Animal collision,Front-end damage,No
4,Someone scraped along the driver's side of my parked car. I noticed the damage when I returned.,Sideswipe collision,Driver-side damage,Unknown
5,I was sitting at a traffic light when a vehicle rear-ended me. The back bumper is damaged and th...,Rear-end collision,Bumper damage,No
6,"Another driver backed into my car outside the supermarket, denting the passenger door. I was abl...",Parking lot collision,Passenger door dent,Yes
7,A car moved into my lane and clipped my mirror. The mirror shattered. I don't know if there are ...,Lane-change collision,Broken side mirror,Unknown
8,"I hit a deer on my way home. My vehicle has front-end damage, but I managed to drive it to my ho...",Animal collision,Front-end damage,Yes
9,A passing vehicle sideswiped the driver's side of my car. The door was damaged and the vehicle h...,Sideswipe collision,Driver-side damage,No


In [38]:
import json

results = []

for _, row in test_df.iterrows():
    response = generate_structured_claim(row["narrative"])

    try:
        prediction = json.loads(response)
        valid_json = isinstance(prediction, dict)
    except (json.JSONDecodeError, TypeError):
        prediction = {}
        valid_json = False

    result = {
        "narrative": row["narrative"],
        "valid_json": valid_json,
        "raw_response": response
    }

    for field in ["collision_type", "damage", "drivable"]:
        expected = row[field]
        predicted = prediction.get(field, "MISSING")

        result[f"expected_{field}"] = expected
        result[f"predicted_{field}"] = predicted
        result[f"correct_{field}"] = (
            isinstance(predicted, str)
            and predicted.strip().lower() == expected.strip().lower()
        )

    results.append(result)

results_df = pd.DataFrame(results)

fields = ["collision_type", "damage", "drivable"]

print("INDEPENDENT TEST RESULTS")
print("=" * 45)

print(f"Valid JSON: {results_df['valid_json'].mean():.1%}")

for field in fields:
    accuracy = results_df[f"correct_{field}"].mean()
    print(f"{field} accuracy: {accuracy:.1%}")

results_df["all_fields_correct"] = results_df[
    [f"correct_{field}" for field in fields]
].all(axis=1)

print(
    "Complete record accuracy:",
    f"{results_df['all_fields_correct'].mean():.1%}"
)

display(
    results_df[
        ["valid_json"] +
        [f"correct_{field}" for field in fields] +
        ["all_fields_correct"]
    ]
)


INDEPENDENT TEST RESULTS
Valid JSON: 100.0%
collision_type accuracy: 30.0%
damage accuracy: 10.0%
drivable accuracy: 70.0%
Complete record accuracy: 0.0%


,valid_json,correct_collision_type,correct_damage,correct_drivable,all_fields_correct
0,True,False,False,True,False
1,True,False,False,False,False
2,True,True,False,True,False
3,True,False,False,True,False
4,True,False,False,False,False
5,True,True,False,True,False
6,True,False,False,True,False
7,True,True,False,False,False
8,True,False,True,True,False
9,True,False,False,True,False


## Independent Test Results and Interpretation

The independent evaluation produced results that were different from what I initially expected based on the training and validation loss. All 10 responses were valid JSON, which suggests that the model learned the required output structure. However, the accuracy of the extracted information was considerably lower.

The model correctly classified the collision type in 30% of the examples and identified the expected damage category in only 10%. Drivability performed better at 70%, but none of the 10 accident narratives had all three fields correct.

I found this interesting because the validation loss had decreased to approximately 0.016 by the end of training. Initially, that seemed like a strong indication that the model was learning successfully. The independent results showed me that predicting familiar training patterns is not the same as accurately handling new situations.

One possible explanation is the way I generated the synthetic dataset. Although I created 75 unique narratives, they were built from a limited number of sentence templates. The model may have learned associations between specific phrases and labels without becoming reliable at recognizing different descriptions of the same accident.

Another limitation is that my evaluation uses exact matching. This means a response with slightly different wording could be counted as incorrect even if the meaning is similar. I need to inspect the individual predictions before deciding whether each mismatch represents a genuine error.

Based on these results, I would not consider the model ready for real insurance claim intake. However, the experiment helped me understand why validation loss alone is not enough to evaluate a fine-tuned language model.

In [39]:
comparison_columns = [
    "expected_collision_type",
    "predicted_collision_type",
    "expected_damage",
    "predicted_damage",
    "expected_drivable",
    "predicted_drivable"
]

pd.set_option("display.max_colwidth", 100)

display(results_df[comparison_columns])

print("\nEXAMPLE OF AN INCORRECT PREDICTION")
print("=" * 50)

incorrect = results_df[
    ~results_df["all_fields_correct"]
].iloc[0]

print("Narrative:", incorrect["narrative"])
print("\nModel response:")
print(incorrect["raw_response"])


,expected_collision_type,predicted_collision_type,expected_damage,predicted_damage,expected_drivable,predicted_drivable
0,Rear-end collision,Lane-change collision,Bumper damage,Cracks in bumper,Yes,Yes
1,Parking lot collision,Rear-end collision,Passenger door dent,Dent,Unknown,No
2,Lane-change collision,Lane-change collision,Broken side mirror,Broken mirror,Yes,Yes
3,Animal collision,Rural road collision,Front-end damage,Front-end collision,No,No
4,Sideswipe collision,Scraped along the driver's side of my parked car,Driver-side damage,I noticed the damage when I returned,Unknown,I didn't see the damage myself
5,Rear-end collision,Rear-end collision,Bumper damage,Back bumper damage,No,No
6,Parking lot collision,Driver-backed collision,Passenger door dent,Dented passenger door,Yes,Yes
7,Lane-change collision,Lane-change collision,Broken side mirror,Mirror shattered,Unknown,No
8,Animal collision,Lane-change collision,Front-end damage,Front-end damage,Yes,Yes
9,Sideswipe collision,Passing vehicle sideswipe collision,Driver-side damage,Oddly damaged door,No,No



EXAMPLE OF AN INCORRECT PREDICTION
Narrative: I was waiting at a stop sign when a truck crashed into the back of my car. My bumper is cracked, but I drove away.

Model response:
{"collision_type": "Lane-change collision", "damage": "Cracks in bumper", "drivable": "Yes"}


## Error Analysis and Evaluation Limitations

After examining the individual predictions, I realized that my initial accuracy scores did not tell the entire story. Some responses were marked incorrect because the model used different wording from my expected labels. For example, it predicted "Cracks in bumper" instead of "Bumper damage" and "Dented passenger door" instead of "Passenger door dent." These descriptions communicate essentially the same information, even though my exact-match evaluation counted them as errors.

However, I also identified genuine mistakes. In one example, the model classified a rear-end collision as a lane-change collision. In another, it described an animal collision as a rural road collision. The model also sometimes confused missing drivability information with a definite answer of "No."

This helped me recognize that evaluating a generative AI model requires more than calculating a single accuracy score. Exact matching is useful when a system must produce standardized categories, but it does not always reflect whether the model understood the underlying information.

For a future version of this project, I would consider evaluating semantic correctness separately from strict category compliance. I would also improve the training dataset by introducing more varied narratives and explicitly teaching the model how to handle missing information.

In [40]:
from transformers import AutoModelForCausalLM
import torch
import json

original_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
).to("cpu")

original_model.eval()

def generate_original_claim(claim_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": claim_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = original_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        response_tokens,
        skip_special_tokens=True
    )

print("Original model loaded for controlled comparison.")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Original model loaded for controlled comparison.


In [41]:
original_results = []

for _, row in test_df.iterrows():
    response = generate_original_claim(row["narrative"])

    try:
        prediction = json.loads(response)
        valid_json = isinstance(prediction, dict)
    except (json.JSONDecodeError, TypeError):
        prediction = {}
        valid_json = False

    result = {"valid_json": valid_json}

    for field in ["collision_type", "damage", "drivable"]:
        predicted = prediction.get(field, "MISSING")
        expected = row[field]

        result[f"correct_{field}"] = (
            isinstance(predicted, str)
            and predicted.strip().lower() == expected.strip().lower()
        )

    original_results.append(result)

original_df = pd.DataFrame(original_results)

print("ORIGINAL MODEL RESULTS")
print("=" * 40)

print(f"Valid JSON: {original_df['valid_json'].mean():.1%}")

for field in ["collision_type", "damage", "drivable"]:
    print(
        f"{field} accuracy:",
        f"{original_df[f'correct_{field}'].mean():.1%}"
    )

original_df["all_fields_correct"] = original_df[
    [
        "correct_collision_type",
        "correct_damage",
        "correct_drivable"
    ]
].all(axis=1)

print(
    "Complete record accuracy:",
    f"{original_df['all_fields_correct'].mean():.1%}"
)


ORIGINAL MODEL RESULTS
Valid JSON: 0.0%
collision_type accuracy: 0.0%
damage accuracy: 0.0%
drivable accuracy: 0.0%
Complete record accuracy: 0.0%


## Comparison of the Original and Fine-Tuned Models

To evaluate whether full fine-tuning improved performance, I loaded a separate copy of the original SmolLM2 model and tested it using the same 10 accident narratives and system prompt as the fine-tuned version.

The original model did not produce valid JSON for any of the test cases. As a result, its structured extraction accuracy was 0% across all three fields. However, this does not necessarily mean the original model failed to understand the accident descriptions. My evaluation required a valid JSON object before individual fields could be scored.

After fine-tuning, the model produced valid JSON for all 10 test cases. Collision type accuracy reached 30%, damage accuracy was 10%, and drivability accuracy was 70%. No complete record contained all three expected values.

These results suggest that full fine-tuning improved the model's ability to follow a structured response format. However, the experiment did not demonstrate reliable extraction accuracy. Some of the incorrect scores were caused by differences in wording, while others reflected genuine classification mistakes.

The biggest lesson I learned from this comparison is that successful fine-tuning does not automatically mean a model is ready for practical use. Training loss, validation loss, formatting accuracy, and factual extraction accuracy each measure different aspects of performance. I would need a larger, more diverse dataset and additional evaluation before considering this application suitable for real insurance claim intake.

In [42]:
import json
import os

save_directory = "/content/claim_intake_fine_tuned"

os.makedirs(save_directory, exist_ok=True)

trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

metrics = {
    "model": model_name,
    "fine_tuning_method": "Full-parameter fine-tuning",
    "total_parameters": 134515008,
    "trainable_parameters": 134515008,
    "training_examples": 60,
    "validation_examples": 15,
    "test_examples": 10,
    "epochs": 3,
    "training_time_minutes": round(elapsed_time / 60, 2),
    "final_training_loss": training_result.training_loss,
    "validation_loss": evaluation_results["eval_loss"],
    "original_valid_json_rate": original_df["valid_json"].mean(),
    "fine_tuned_valid_json_rate": results_df["valid_json"].mean(),
    "fine_tuned_collision_accuracy": results_df["correct_collision_type"].mean(),
    "fine_tuned_damage_accuracy": results_df["correct_damage"].mean(),
    "fine_tuned_drivable_accuracy": results_df["correct_drivable"].mean(),
    "fine_tuned_complete_record_accuracy": results_df["all_fields_correct"].mean()
}

with open(
    os.path.join(save_directory, "evaluation_metrics.json"),
    "w"
) as file:
    json.dump(metrics, file, indent=4)

results_df.to_csv(
    os.path.join(save_directory, "test_predictions.csv"),
    index=False
)

print("Model and evaluation results saved!")
print("Location:", save_directory)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and evaluation results saved!
Location: /content/claim_intake_fine_tuned


## Final Evaluation and Lessons Learned

For this milestone, I fully fine-tuned SmolLM2-135M-Instruct in Google Colab. I switched to this model after Milestone 2 feedback suggested the OpenAI model I’d been testing might not support the kind of fine-tuning the project needed. The assignment originally pointed to OpenAI, but an open-weight model let me train every parameter directly, without LoRA.

I trained on 60 examples, validated on 15, and held out 10 separate narratives for testing. Training took over a minute (1.45 minutes) across three epochs, and both training and validation loss went down. At first, that looked like the model was learning the task.

The test set told a different story. On format, the results were clear: the fine-tuned model produced valid JSON for all 10 test cases, and the original model produced none. On accuracy, it was much weaker. Exact-match accuracy was 30% for collision type, 10% for damage, and 70% for drivability, and not a single record got all three fields right.

Going through the predictions one by one, I found two kinds of misses. Some answers were reasonable in meaning but worded differently from my expected categories. Others were real mistakes, such as wrong collision types and claims about drivability that the narrative didn’t support.

The biggest lesson is that a model that trains well isn’t necessarily one you can rely on. Validation loss dropped substantially, yet the model struggled once the accident descriptions strayed from the templates it was trained on.

If I kept working on this, I would add more varied accident narratives, set stricter labeling rules, and score both exact-category accuracy and semantic correctness. I would also test on a much larger set of unfamiliar examples before trusting it for any real use.

This project taught me how full fine-tuning works and why evaluation has to go beyond training loss.

In [43]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    "/content/claim_intake_fine_tuned",
    "zip",
    "/content/claim_intake_fine_tuned"
)

print("Archive created:", archive_path)
files.download(archive_path)


Archive created: /content/claim_intake_fine_tuned.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>